<a href="https://colab.research.google.com/github/OmkarRameshJadhav108/-OpenEdHub-Open-Source-Online-Learning-Platform/blob/main/bubblermerge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%%writefile program.cpp
#include <iostream>
#include <vector>
#include <cstdlib>
#include <ctime>
#include <omp.h>

using namespace std;

#define SIZE 10000

// Sequential Bubble Sort
void bubbleSortSeq(vector<int>& arr) {
    int n = arr.size();
    for (int i = 0; i < n - 1; i++)
        for (int j = 0; j < n - i - 1; j++)
            if (arr[j] > arr[j + 1])
                swap(arr[j], arr[j + 1]);
}

// Parallel Bubble Sort (Odd-Even)
void bubbleSortParallel(vector<int>& arr) {
    int n = arr.size();

    for (int i = 0; i < n; i++) {
        #pragma omp parallel for
        for (int j = 0; j < n - 1; j += 2)
            if (arr[j] > arr[j + 1])
                swap(arr[j], arr[j + 1]);

        #pragma omp parallel for
        for (int j = 1; j < n - 1; j += 2)
            if (arr[j] > arr[j + 1])
                swap(arr[j], arr[j + 1]);
    }
}

// Merge function
void merge(vector<int>& arr, int l, int m, int r) {
    vector<int> left(arr.begin() + l, arr.begin() + m + 1);
    vector<int> right(arr.begin() + m + 1, arr.begin() + r + 1);

    int i = 0, j = 0, k = l;

    while (i < left.size() && j < right.size())
        arr[k++] = (left[i] <= right[j]) ? left[i++] : right[j++];

    while (i < left.size()) arr[k++] = left[i++];
    while (j < right.size()) arr[k++] = right[j++];
}

// Sequential Merge Sort
void mergeSortSeq(vector<int>& arr, int l, int r) {
    if (l < r) {
        int m = (l + r) / 2;
        mergeSortSeq(arr, l, m);
        mergeSortSeq(arr, m + 1, r);
        merge(arr, l, m, r);
    }
}

// Parallel Merge Sort
void mergeSortParallel(vector<int>& arr, int l, int r, int depth) {
    if (l < r) {
        int m = (l + r) / 2;

        if (depth <= 0) {
            mergeSortSeq(arr, l, m);
            mergeSortSeq(arr, m + 1, r);
        } else {
            #pragma omp parallel sections
            {
                #pragma omp section
                mergeSortParallel(arr, l, m, depth - 1);

                #pragma omp section
                mergeSortParallel(arr, m + 1, r, depth - 1);
            }
        }

        merge(arr, l, m, r);
    }
}

// Generate random array
void generateRandom(vector<int>& arr) {
    for (int& x : arr)
        x = rand() % 100000;
}

// MAIN
int main() {
    vector<int> arr(SIZE), temp;

    srand(time(0));
    generateRandom(arr);

    double start, end;

    temp = arr;
    start = omp_get_wtime();
    bubbleSortSeq(temp);
    end = omp_get_wtime();
    cout << "Sequential Bubble Sort: " << end - start << " sec\n";

    temp = arr;
    start = omp_get_wtime();
    bubbleSortParallel(temp);
    end = omp_get_wtime();
    cout << "Parallel Bubble Sort: " << end - start << " sec\n";

    temp = arr;
    start = omp_get_wtime();
    mergeSortSeq(temp, 0, SIZE - 1);
    end = omp_get_wtime();
    cout << "Sequential Merge Sort: " << end - start << " sec\n";

    temp = arr;
    start = omp_get_wtime();
    mergeSortParallel(temp, 0, SIZE - 1, 4);
    end = omp_get_wtime();
    cout << "Parallel Merge Sort: " << end - start << " sec\n";

    return 0;
}

Overwriting program.cpp


/usr/bin/ld: /tmp/cc95a93F.o: in function `main':
program.cpp:(.text+0x864): undefined reference to `omp_get_wtime'
/usr/bin/ld: program.cpp:(.text+0x87e): undefined reference to `omp_get_wtime'
/usr/bin/ld: program.cpp:(.text+0x8ec): undefined reference to `omp_get_wtime'
/usr/bin/ld: program.cpp:(.text+0x906): undefined reference to `omp_get_wtime'
/usr/bin/ld: program.cpp:(.text+0x974): undefined reference to `omp_get_wtime'
/usr/bin/ld: /tmp/cc95a93F.o:program.cpp:(.text+0x998): more undefined references to `omp_get_wtime' follow
collect2: error: ld returned 1 exit status


In [7]:
!g++ program.cpp -fopenmp -o program

In [8]:
!./program

Sequential Bubble Sort: 0.775488 sec
Parallel Bubble Sort: 0.840025 sec
Sequential Merge Sort: 0.00814271 sec
Parallel Merge Sort: 0.00635287 sec
